# Results from CSV (match `check.ipynb` layout)

Reads `base.csv` / `ensemble.csv` / `tabm.csv` per dataset under `RESULTS_ROOT/{dataset_dir}/`.

For each **dataset**, **method** (BASE / ENS / TABM), **architecture** (`model`), and **split**:
- Among all rows (all `num_layers` × `hidden_dim` × `lr`), pick the row with **highest `val_metric`** (same objective as training / early stopping).
- Report **test** score: **`test_roc_auc`** for `questions` / `minesweeper` / `tolokers`, else **`test_acc`** (same rule as `check.ipynb`).

Then **mean ± std** over splits (0–9) and pivot like `check.ipynb`.

In [4]:
import pandas as pd
from pathlib import Path
from typing import Optional

# Root folder containing per-dataset subdirs with base.csv, ensemble.csv, tabm.csv
RESULTS_ROOT = Path('results_simple_test')
# e.g. simple runs:
# RESULTS_ROOT = Path('results_simple_test')

models = ['GT-sep', 'GAT-sep', 'GAT', 'TAG', 'GCN', 'SAGE', 'GT', 'ResNet']

ROC_PRIMARY_DATASETS = frozenset({'questions', 'minesweeper', 'tolokers'})

# Keys = dataset id (as in check.ipynb); values = subdir name under RESULTS_ROOT
DATASET_DIRS = {
    # 'roman_empire': 'roman_empire',
    # 'amazon_ratings': 'amazon_ratings',
    # 'minesweeper': 'minesweeper',
    'questions': 'questions',
    'tolokers': 'tolokers',
}

METHOD_FILES = {
    'BASE': 'base.csv',
    'ENS': 'ensemble.csv',
    'TABM': 'tabm.csv',
}

NUM_SPLITS = 10


def primary_test_value(row: pd.Series, primary_is_roc: bool) -> Optional[float]:
    """Test metric to aggregate (aligned with check.ipynb)."""
    if primary_is_roc:
        v = row.get('test_roc_auc')
        if pd.notna(v):
            return float(v)
        return None
    v = row.get('test_acc')
    if pd.notna(v):
        return float(v)
    return None


def best_row_per_split(df: pd.DataFrame, model: str, split: int) -> Optional[pd.Series]:
    """Single best config for (model, split) by validation metric."""
    sub = df[(df['model'] == model) & (df['split'] == split)].copy()
    if sub.empty:
        return None
    sub = sub.dropna(subset=['val_metric'])
    if sub.empty:
        return None
    i = sub['val_metric'].idxmax()
    return sub.loc[i]


rows = []
for dataset_name, ds_dir in DATASET_DIRS.items():
    primary_is_roc = dataset_name in ROC_PRIMARY_DATASETS
    for method_name, csv_name in METHOD_FILES.items():
        csv_path = RESULTS_ROOT / ds_dir / csv_name
        if not csv_path.is_file():
            continue
        df = pd.read_csv(csv_path)
        if 'val_metric' not in df.columns:
            raise ValueError(f"Missing val_metric column in {csv_path}")

        for model in models:
            split_scores = []
            for split in range(NUM_SPLITS):
                row = best_row_per_split(df, model, split)
                if row is None:
                    continue
                pv = primary_test_value(row, primary_is_roc)
                if pv is not None:
                    split_scores.append(pv)

            if split_scores:
                rows.append({
                    'model_variant': f'{model}_{method_name}',
                    'dataset': dataset_name,
                    'mean_acc': float(pd.Series(split_scores).mean()),
                    'std_acc': float(pd.Series(split_scores).std(ddof=0)),
                })

if not rows:
    raise RuntimeError(
        f"No data found under {RESULTS_ROOT}. "
        "Set RESULTS_ROOT and ensure CSVs exist (base.csv / ensemble.csv / tabm.csv per dataset)."
    )

df_rows = pd.DataFrame(rows)

_col_order = list(DATASET_DIRS.keys())
mx_mean = df_rows.pivot_table(
    index='model_variant',
    columns='dataset',
    values='mean_acc',
    aggfunc='mean',
).sort_index().mul(100)
mx_mean = mx_mean.reindex(columns=[c for c in _col_order if c in mx_mean.columns])

mx_std = df_rows.pivot_table(
    index='model_variant',
    columns='dataset',
    values='std_acc',
    aggfunc='mean',
).reindex(mx_mean.index).mul(100)
mx_std = mx_std.reindex(columns=mx_mean.columns)

mx_text = mx_mean.round(3).astype(str) + ' ± ' + mx_std.round(3).astype(str)


def highlight_top2(col):
    styles = ['' for _ in col]
    non_na = col.dropna()
    if non_na.empty:
        return styles
    unique_vals = sorted(non_na.unique(), reverse=True)
    best = unique_vals[0]
    second = unique_vals[1] if len(unique_vals) > 1 else None
    for i, v in enumerate(col):
        if pd.isna(v):
            continue
        if v == best:
            styles[i] = 'background-color: #ffd700; font-weight: 700'
        elif second is not None and v == second:
            styles[i] = 'background-color: #c0c0c0; font-weight: 700'
    return styles


mx_text.style.apply(lambda col: highlight_top2(mx_mean[col.name]), axis=0)

dataset,questions,tolokers
model_variant,,
ResNet_BASE,71.229 ± 0.0,72.125 ± 0.0
ResNet_ENS,70.656 ± 0.0,72.118 ± 0.0
ResNet_TABM,70.543 ± 0.0,71.521 ± 0.0


In [ ]:
# Long table (optional): one row per dataset × method × model with mean/std over splits
df_rows.sort_values(['dataset', 'model_variant']).reset_index(drop=True)